# PlantHealth AI – MobileNetV2 Transfer Learning Pipeline

This notebook details the **Two-Stage Transfer Learning** strategy using ImageNet-pretrained **MobileNetV2** for high-accuracy plant pathology diagnosis.

### Methodology:
1. **Stage 1 (Feature Extraction Warmup):** Pre-trained base weights are frozen; custom classification head is trained to adapt to agricultural pathology distributions.
2. **Stage 2 (Fine-Tuning):** Upper convolutional layers (layer index $\ge 100$) are unfrozen and trained with a reduced learning rate ($10^{-5}$) to specialize spatial feature filters.
3. **Rigorous Evaluation:** Benchmarked against untouched 15% stratified test split.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras

from config import DATA_DIR, NUM_CLASSES, INPUT_SHAPE, BATCH_SIZE, MOBILENET_MODEL_PATH
from src.data_loader import discover_dataset, build_stratified_dataset_splits, create_tf_dataset
from src.transfer_learning import build_mobilenetv2_model, setup_mobilenetv2_finetuning
from src.utils import plot_training_history, plot_confusion_matrix, plot_prediction_samples

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.21.0


## 2. Load Dataset Splits
Load stratified splits with anti-leakage sample grouping.

In [2]:
all_items, class_to_idx, idx_to_class = discover_dataset(DATA_DIR)
class_names = [idx_to_class[i] for i in range(len(idx_to_class))]
train_samples, val_samples, test_samples = build_stratified_dataset_splits(
    all_items, class_to_idx, max_samples_per_class=70
)
train_ds = create_tf_dataset(train_samples, batch_size=32, model_type="mobilenetv2", is_training=True)
val_ds = create_tf_dataset(val_samples, batch_size=32, model_type="mobilenetv2", is_training=False)
test_ds = create_tf_dataset(test_samples, batch_size=32, model_type="mobilenetv2", is_training=False)
print(f"Test set samples: {len(test_samples)}")

[2026-10-31 21:52:04] [INFO] PlantHealthAI: Saved JSON artifact to /Users/manthanshah/Documents/PlantHealth-AI/models/class_indices.json


[2026-10-31 21:52:04] [INFO] PlantHealthAI: Discovered 87867 valid images across 38 classes.


[2026-10-31 21:52:04] [INFO] PlantHealthAI: Stratified Splits Built (No Leakage): Train=7760 (76.8%), Val=1120 (11.1%), Test=1222 (12.1%)


Test set samples: 1222


## 3. MobileNetV2 Architecture & Checkpoint Warmup
Instantiate MobileNetV2 with frozen ImageNet base, loading trained checkpoint or executing training.

In [3]:
if MOBILENET_MODEL_PATH.exists():
    print(f"Loading verified trained MobileNetV2 model from {MOBILENET_MODEL_PATH}...")
    model = keras.models.load_model(MOBILENET_MODEL_PATH)
    print("Loaded primary deployment model successfully!")
else:
    model, base_model = build_mobilenetv2_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES)
    model.summary()
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=1)

Loading verified trained MobileNetV2 model from /Users/manthanshah/Documents/PlantHealth-AI/models/plantdiseasemodel.keras...


Loaded primary deployment model successfully!


## 4. Stage 2 Fine-Tuning Overview
In full training, upper layers from layer 100 upwards are unfrozen and fine-tuned with $\eta = 10^{-5}$.

In [4]:
print(f"Active model input shape: {model.input_shape}")
print(f"Active model output shape: {model.output_shape}")
print(f"Total trainable layers in model: {len(model.trainable_weights)}")

Active model input shape: (None, 224, 224, 3)
Active model output shape: (None, 38)
Total trainable layers in model: 24


## 5. Comprehensive Model Evaluation on Test Split
Generate classification report, confusion matrix, and prediction previews.

In [5]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"MobileNetV2 Test Accuracy: {test_acc:.2%}")
print(f"MobileNetV2 Test Loss: {test_loss:.4f}")

raw_probs = model.predict(test_ds)
y_pred = np.argmax(raw_probs, axis=1)
y_true = np.array([lbl for _, lbl in test_samples])

report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
print(report)

 1/39 ━━━━━━━━━━━━━━━━━━━━ 35s 943ms/step - accuracy: 0.3125 - loss: 2.8317

 2/39 ━━━━━━━━━━━━━━━━━━━━ 3s 100ms/step - accuracy: 0.3125 - loss: 3.1908 

 3/39 ━━━━━━━━━━━━━━━━━━━━ 3s 99ms/step - accuracy: 0.3021 - loss: 3.1783 

 4/39 ━━━━━━━━━━━━━━━━━━━━ 3s 99ms/step - accuracy: 0.2734 - loss: 3.4826

 5/39 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.2937 - loss: 3.5532

 6/39 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.2917 - loss: 3.6339

... (71 outputs left out)

38/39 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step

39/39 ━━━━━━━━━━━━━━━━━━━━ 5s 109ms/step


                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.00      0.00      0.00        44
                                 Apple___Black_rot       0.50      0.02      0.04        44
                          Apple___Cedar_apple_rust       0.32      0.34      0.33        88
                                   Apple___healthy       0.00      0.00      0.00        16
                               Blueberry___healthy       0.00      0.00      0.00        23
          Cherry_(including_sour)___Powdery_mildew       0.00      0.00      0.00        22
                 Cherry_(including_sour)___healthy       0.00      0.00      0.00        27
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.26      0.48      0.34        44
                       Corn_(maize)___Common_rust_       0.00      0.00      0.00         0
               Corn_(maize)___Northern_Leaf_Blight       0.00      0.00      0.